<a href="https://colab.research.google.com/github/jorobledo/blcourse_test/blob/main/BLcourse4/colab/BNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Initialize Colab
!git clone https://github.com/jorobledo/bayesian_learning_course.git
%cd bayesian_learning_course/BLcourse4/colab/
!pip install -q bayesian_torch

# Probabilistic Bayesian Neural Networks

**Author:** [Jose Robledo](https://jorobledo.github.io/)<br>
**Date created:** 20.01.2026<br>
**Description:** Building probabilistic Bayesian neural network models with Bayesian-torch.

## Introduction

This notebook is a Bayesian-torch adaptation of the [notebook from Khalid Salama](https://github.com/ksalama/keras-io/blob/2549b0afb720f9b6f7e3b6c82dcb456472101539/examples/keras_recipes/ipynb/bayesian_neural_networks.ipynb#L9).

Taking a probabilistic approach to deep learning allows to account for *uncertainty*,
so that models can assign less levels of confidence to incorrect predictions.
Sources of uncertainty can be found in the data, due to measurement error or
noise in the labels, or the model, due to insufficient data availability for
the model to learn effectively.


This example demonstrates how to build basic probabilistic Bayesian neural networks
to account for these two types of uncertainty.
We will use [Bayesian-Torch](https://github.com/IntelLabs/bayesian-torch) library.

You can install `Bayesian-Torch` using the following command:

```bash
pip install bayesian-torch
```

## The dataset

We use the [Wine Quality](https://archive.ics.uci.edu/ml/datasets/wine+quality)
dataset.


We use the white wine subset, which contains 4,898 examples.
The dataset has 11 numerical physicochemical features of the wine, and the task
is to predict the wine quality, which is a score between 0 and 10.
In this example, we treat this as a regression task.

## Setup

In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.optim import RMSprop
import torch.nn.functional as F
from torch.distributions import Normal, Independent
from torch.utils.data import TensorDataset, DataLoader

from bayesian_torch.layers import LinearReparameterization

## Create training and evaluation datasets

Here, we load the `wine_quality` dataset using `tfds.load()`, and we convert
the target feature to float. Then, we shuffle the dataset and split it into
training and test sets. We take the first `train_size` examples as the train
split, and the rest as the test split.

In [2]:
def get_train_and_test_dataloaders(
    train_size: float,
    batch_size: int,
    seed: int = 0,
    num_workers: int = 0,
    pin_memory: bool = True,
    dataset_size: float = 1.0
):
    """
    train_size: float in (0, 1], proportion of data used for training (e.g. 0.8)
    batch_size: batch size for both DataLoaders
    seed: RNG seed for reproducible shuffling (train split only)
    """

    if not (0.0 < train_size <= 1.0):
        raise ValueError(f"train_size must be in (0, 1], got {train_size}")

    url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
    df = pd.read_csv(url_red, sep=";")

    # df = pd.read_csv("/home/jorobledo/Downloads/tf_wine_dataset.csv").drop(columns=["Unnamed: 0"])

    if "quality" not in df.columns:
        raise ValueError("Expected target column 'quality'.")

    # Features / target
    if dataset_size < 1.0:
        n_samples = int(len(df) * dataset_size)
        df = df.sample(n=n_samples, random_state=seed).reset_index(drop=True)
    X = df.drop(columns=["quality"]).to_numpy(dtype=np.float32)
    y = df["quality"].to_numpy(dtype=np.float32)

    n = len(df)
    n_train = int(train_size * n)

    if n_train == 0 or n_train == n:
        raise ValueError(
            f"train_size={train_size} results in {n_train} training samples."
        )

    # --- Split FIRST (same as dataset.take / skip) ---
    X_train, y_train = X[:n_train], y[:n_train]
    X_test, y_test = X[n_train:], y[n_train:]

    # --- Shuffle training only ---
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_train)
    X_train, y_train = X_train[perm], y_train[perm]

    # --- Convert to torch tensors ---
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    X_test_t = torch.from_numpy(X_test)
    y_test_t = torch.from_numpy(y_test)

    train_ds = TensorDataset(X_train_t, y_train_t)
    test_ds = TensorDataset(X_test_t, y_test_t)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,  # already shuffled above
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, test_loader


## Compile, train, and evaluate the model

In [3]:
learning_rate = 0.001


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    num_samples = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            preds = model(X).squeeze()
            loss = loss_fn(preds, y)

            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            num_samples += batch_size

    mse = total_loss / num_samples
    return np.sqrt(mse)

def run_experiment(model, loss_fn, train_dataloader, test_dataloader, num_epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = RMSprop(model.parameters(), lr=learning_rate)

    print("Start training the model...")

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_samples = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            preds = model(X).squeeze()
            loss = loss_fn(preds, y)
            loss.backward()
            optimizer.step()

            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            num_samples += batch_size

        # Optional: validation each epoch (like Keras `validation_data`)
        val_rmse = evaluate(model, test_dataloader, loss_fn, device)

    print("Model training finished.")

    train_rmse = evaluate(model, train_dataloader, loss_fn, device)
    print(f"Train RMSE: {round(train_rmse, 3)}")

    print("Evaluating model performance...")
    test_rmse = evaluate(model, test_dataloader, loss_fn, device)
    print(f"Test RMSE: {round(test_rmse, 3)}")


## Create model inputs

In [4]:
FEATURE_NAMES = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

NUM_FEATURES = len(FEATURE_NAMES)


## Experiment 1: standard neural network

We create a standard deterministic neural network model as a baseline.

In [5]:
hidden_units = [8, 8]
class BaselineModel(nn.Module):
    def __init__(self):
        super().__init__()

        layers = []

        # Equivalent to keras.layers.concatenate + BatchNormalization
        layers.append(nn.BatchNorm1d(NUM_FEATURES))

        input_dim = NUM_FEATURES
        for units in hidden_units:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.Sigmoid())
            input_dim = units

        # Output layer: single deterministic point estimate
        layers.append(nn.Linear(input_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor):
        # x shape: (batch_size, 11)
        return self.net(x)

Let's split the wine dataset into training and test sets, with 85% and 15% of
the examples, respectively.

In [6]:
train_size = 0.85
batch_size=256
train_dataloader, test_dataloader = get_train_and_test_dataloaders(train_size, batch_size)
len(train_dataloader.dataset)

4163

Now let's train the baseline model. We use the `MeanSquaredError`
as the loss function.

In [7]:
num_epochs = 100
mse_loss = nn.MSELoss()

baseline_model = BaselineModel()
run_experiment(
    model=baseline_model,
    loss_fn=mse_loss,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
)

Start training the model...
Model training finished.
Train RMSE: 0.775
Evaluating model performance...
Test RMSE: 0.685


We take a sample from the test set use the model to obtain predictions for them.
Note that since the baseline model is deterministic, we get a single a
*point estimate* prediction for each test example, with no information about the
uncertainty of the model nor the prediction.

In [8]:
sample = 10

baseline_model.eval()

# Take a single batch from the test DataLoader
examples, targets = next(iter(test_dataloader))

device = next(baseline_model.parameters()).device  # cuda or cpu

# Keep only `sample` examples
examples = examples[:sample].to(device)
targets = targets[:sample].to(device)

with torch.no_grad():
    predicted = baseline_model(examples)

for i in range(sample):
    print(
        f"Predicted: {round(predicted[i].item(), 1)} - "
        f"Actual: {targets[i].item()}"
    )


Predicted: 5.8 - Actual: 5.0
Predicted: 5.8 - Actual: 5.0
Predicted: 6.4 - Actual: 7.0
Predicted: 6.4 - Actual: 7.0
Predicted: 6.6 - Actual: 8.0
Predicted: 6.5 - Actual: 6.0
Predicted: 6.5 - Actual: 7.0
Predicted: 5.6 - Actual: 7.0
Predicted: 6.4 - Actual: 5.0
Predicted: 6.2 - Actual: 6.0


## Experiment 2: Bayesian neural network (BNN)

The object of the Bayesian approach for modeling neural networks is to capture
the *epistemic uncertainty*, which is uncertainty about the model fitness,
due to limited training data.

The idea is that, instead of learning specific weight (and bias) *values* in the
neural network, the Bayesian approach learns weight *distributions*
- from which we can sample to produce an output for a given input -
to encode weight uncertainty.

Thus, we need to define prior and the posterior distributions of these weights,
and the training process is to learn the parameters of these distributions.

In [9]:
NUM_FEATURES = 11
hidden_units = [8, 8]

# Prior / posterior init hyperparams (common defaults in bayesian-torch examples)
PRIOR_MU = 0.0
PRIOR_VAR = 1.0
POSTERIOR_MU_INIT = 0.0
POSTERIOR_RHO_INIT = -3.0  # smaller => smaller initial sigma

class BayesianWineMLP(nn.Module):
    def __init__(
        self,
        prior_mean=0.0,
        prior_variance=1.0,
        posterior_mu_init=0.0,
        posterior_rho_init=-3.0,  # smaller => smaller initial sigma
        activation="sigmoid",
    ):
        super().__init__()

        self.bn = nn.BatchNorm1d(NUM_FEATURES, eps=1e-3, momentum=0.99)

        self.fc1 = LinearReparameterization(
            in_features=NUM_FEATURES,
            out_features=hidden_units[0],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.fc2 = LinearReparameterization(
            in_features=hidden_units[0],
            out_features=hidden_units[1],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.out = nn.Linear(
            in_features=hidden_units[1],
            out_features=1
        )

        if activation == "sigmoid":
            self.act = nn.Sigmoid()
        elif activation == "relu":
            self.act = nn.ReLU()
        else:
            raise ValueError("activation must be 'sigmoid' or 'relu'.")

    def forward(self, x):
        kl_sum = 0.0

        x = self.bn(x)

        x, kl = self.fc1(x)
        kl_sum = kl_sum + kl
        x = self.act(x)

        x, kl = self.fc2(x)
        kl_sum = kl_sum + kl
        x = self.act(x)

        x = self.out(x)
        
        return x, kl_sum

The epistemic uncertainty can be reduced as we increase the size of the
training data. That is, the more data the BNN model sees, the more it is certain
about its estimates for the weights (distribution parameters).
Let's test this behaviour by training the BNN model on a small subset of
the training set, and then on the full training set, to compare the output variances.

### Train BNN  with a small training subset.

In [52]:
train_size = 0.85
batch_size=256
dataset_size = 0.3
small_train_dataloader, small_test_dataloader = get_train_and_test_dataloaders(train_size,
                                                                batch_size,
                                                                dataset_size=dataset_size)
len(small_train_dataloader.dataset)

1248

In [53]:
def evaluate_rmse_bnn(model, dataloader, device):
    model.eval()
    se_sum = 0.0
    n = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            preds, _ = model(X)
            preds = preds.squeeze(-1)

            se_sum += torch.sum((preds - y) ** 2).item()
            n += y.numel()

    return np.sqrt(se_sum / n)

def train_bnn(
    model,
    train_dataloader,
    test_dataloader,
    num_epochs=100,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
    mse_loss = nn.MSELoss()

    # Practical default: scale KL by number of training samples.
    n_train = len(train_dataloader.dataset)
    beta = kl_weight / n_train

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_mse = 0.0
        epoch_kl_term = 0.0
        n_seen = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            preds, kl = model(X)
            preds = preds.squeeze(-1)

            data_loss = mse_loss(preds, y)
            loss = data_loss + beta * kl

            loss.backward()
            optimizer.step()

            bs = y.numel()
            epoch_loss += loss.item() * bs
            epoch_mse += data_loss.item() * bs
            epoch_kl_term += (beta * kl).item() * bs
            n_seen += bs

        if (epoch % print_every == 0) or (epoch == 1) or (epoch == num_epochs):
            train_rmse = evaluate_rmse_bnn(model, train_dataloader, device)
            test_rmse = evaluate_rmse_bnn(model, test_dataloader, device)

            print(
                f"Epoch {epoch:3d}/{num_epochs} | "
                f"loss={epoch_loss/n_seen:.4f} | "
                f"mse={epoch_mse/n_seen:.4f} | "
                f"kl={epoch_kl_term/n_seen:.4f} | "
                f"train_rmse={train_rmse:.3f} | test_rmse={test_rmse:.3f}"
            )

    return model

num_epochs = 500

bnn_model_2_small = BayesianWineMLP(activation="sigmoid", posterior_rho_init=0)
train_bnn(
    model=bnn_model_2_small,
    train_dataloader=small_train_dataloader,
    test_dataloader=small_test_dataloader,
    num_epochs=num_epochs,
    learning_rate=0.001,
    kl_weight=1.0,
    print_every=100,
)


Epoch   1/500 | loss=39.7384 | mse=39.7380 | kl=0.0004 | train_rmse=6.058 | test_rmse=6.053
Epoch 100/500 | loss=6.3151 | mse=6.3145 | kl=0.0006 | train_rmse=2.352 | test_rmse=2.541
Epoch 200/500 | loss=0.8132 | mse=0.8123 | kl=0.0008 | train_rmse=0.907 | test_rmse=0.752
Epoch 300/500 | loss=0.7556 | mse=0.7547 | kl=0.0009 | train_rmse=0.862 | test_rmse=0.794
Epoch 400/500 | loss=0.7088 | mse=0.7078 | kl=0.0010 | train_rmse=0.840 | test_rmse=0.733
Epoch 500/500 | loss=0.7022 | mse=0.7011 | kl=0.0012 | train_rmse=0.851 | test_rmse=0.684


BayesianWineMLP(
  (bn): BatchNorm1d(11, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
  (fc1): LinearReparameterization()
  (fc2): LinearReparameterization()
  (out): Linear(in_features=8, out_features=1, bias=True)
  (act): Sigmoid()
)

Since we have trained a BNN model, the model produces a different output each time
we call it with the same input, since each time a new set of weights are sampled
from the distributions to construct the network and produce an output.
The less certain the mode weights are, the more variability (wider range) we will
see in the outputs of the same inputs.

In [54]:
def mc_predict(model, X, mc_samples=200):
    device = next(model.parameters()).device
    X = X.to(device)
    model.train()

    preds = []
    with torch.no_grad():
        for _ in range(mc_samples):
            yhat, _ = model(X)
            preds.append(yhat.squeeze(-1))

    # (mc_samples, batch)
    return torch.stack(preds, dim=0)

def comput_predictions(bnn_model, test_dataloader):
    Xb, yb = next(iter(test_dataloader))
    mc_preds = mc_predict(bnn_model, Xb[:10], mc_samples=200)

    for i in range(mc_preds.shape[1]):
        p = mc_preds[:, i]
        mean = p.mean().item()
        min_ = p.min().item()
        max_ = p.max().item()
        range_ = max_ - min_
        actual = yb[i].item()

        print(
            f"Predictions mean: {mean:.2f}, "
            f"min: {min_:.2f}, "
            f"max: {max_:.2f}, "
            f"range: {range_:.2f} - "
            f"Actual: {actual:.1f}"
        )

comput_predictions(bnn_model_2_small, small_test_dataloader)


Predictions mean: 6.19, min: 5.74, max: 6.34, range: 0.60 - Actual: 7.0
Predictions mean: 5.65, min: 4.77, max: 6.10, range: 1.33 - Actual: 5.0
Predictions mean: 6.16, min: 5.68, max: 6.33, range: 0.65 - Actual: 6.0
Predictions mean: 5.91, min: 5.30, max: 6.28, range: 0.97 - Actual: 5.0
Predictions mean: 5.63, min: 4.89, max: 6.16, range: 1.27 - Actual: 6.0
Predictions mean: 5.80, min: 5.07, max: 6.25, range: 1.18 - Actual: 5.0
Predictions mean: 5.91, min: 5.11, max: 6.30, range: 1.20 - Actual: 7.0
Predictions mean: 6.07, min: 5.52, max: 6.33, range: 0.81 - Actual: 6.0
Predictions mean: 5.72, min: 5.05, max: 6.14, range: 1.09 - Actual: 5.0
Predictions mean: 6.08, min: 5.47, max: 6.32, range: 0.86 - Actual: 6.0


In [55]:
num_epochs = 500

bnn_model_2_large = BayesianWineMLP(activation="sigmoid", posterior_rho_init=0)
train_bnn(
    model=bnn_model_2_large,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
    learning_rate=0.001,
    kl_weight=1.0,
    print_every=100,
)

comput_predictions(bnn_model_2_large, test_dataloader)


Epoch   1/500 | loss=29.0398 | mse=29.0397 | kl=0.0001 | train_rmse=5.248 | test_rmse=5.183
Epoch 100/500 | loss=0.6798 | mse=0.6795 | kl=0.0003 | train_rmse=0.832 | test_rmse=0.725
Epoch 200/500 | loss=0.6258 | mse=0.6254 | kl=0.0004 | train_rmse=0.792 | test_rmse=0.699
Epoch 300/500 | loss=0.6078 | mse=0.6072 | kl=0.0006 | train_rmse=0.783 | test_rmse=0.715
Epoch 400/500 | loss=0.5986 | mse=0.5978 | kl=0.0008 | train_rmse=0.774 | test_rmse=0.697
Epoch 500/500 | loss=0.5835 | mse=0.5825 | kl=0.0009 | train_rmse=0.766 | test_rmse=0.696
Predictions mean: 5.20, min: 4.91, max: 5.40, range: 0.49 - Actual: 5.0
Predictions mean: 5.16, min: 4.87, max: 5.37, range: 0.50 - Actual: 5.0
Predictions mean: 5.87, min: 5.62, max: 6.06, range: 0.44 - Actual: 7.0
Predictions mean: 6.13, min: 5.90, max: 6.31, range: 0.41 - Actual: 7.0
Predictions mean: 6.65, min: 6.52, max: 6.70, range: 0.18 - Actual: 8.0
Predictions mean: 6.51, min: 6.29, max: 6.62, range: 0.33 - Actual: 6.0
Predictions mean: 6.53, mi

## Experiment 3: probabilistic Bayesian neural network

So far, the output of the standard and the Bayesian NN models that we built is
deterministic, that is, produces a point estimate as a prediction for a given example.
We can create a probabilistic NN by letting the model output a distribution.
In this case, the model captures the *aleatoric uncertainty* as well,
which is due to irreducible noise in the data, or to the stochastic nature of the
process generating the data.

In this example, we model the output as a `Normal` distribution,
with learnable mean and variance parameters. If the task was classification,
we would have used `IndependentBernoulli` with binary classes, and `OneHotCategorical`
with multiple classes, to model distribution of the model output.

In [56]:
NUM_FEATURES = 11
hidden_units = [8, 8]

class ProbabilisticBayesianWineMLP(nn.Module):
    """
    Outputs an Independent Normal distribution:
      y ~ Normal(loc=mu(x), scale=sigma(x))

    Bayesian weights (epistemic) via bayesian-torch layers.
    Aleatoric via learned sigma(x).
    """
    def __init__(
        self,
        prior_mean=0.0,
        prior_variance=1.0,
        posterior_mu_init=0.0,
        posterior_rho_init=-1.0,
        activation="sigmoid",
        min_sigma=1e-3,   # numerical stability floor
    ):
        super().__init__()
        self.min_sigma = min_sigma

        self.bn = nn.BatchNorm1d(NUM_FEATURES, eps=1e-3, momentum=0.99)

        self.fc1 = LinearReparameterization(
            in_features=NUM_FEATURES,
            out_features=hidden_units[0],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.fc2 = LinearReparameterization(
            in_features=hidden_units[0],
            out_features=hidden_units[1],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )

        self.param_head = nn.Linear(hidden_units[1], 2)

        if activation == "sigmoid":
            self.act = nn.Sigmoid()
        elif activation == "relu":
            self.act = nn.ReLU()
        else:
            raise ValueError("activation must be 'sigmoid' or 'relu'.")

    def forward(self, x):
        kl_sum = 0.0

        x = self.bn(x)

        x, kl = self.fc1(x); kl_sum = kl_sum + kl
        x = self.act(x)

        x, kl = self.fc2(x); kl_sum = kl_sum + kl
        x = self.act(x)

        params = self.param_head(x)          # (batch, 2)
        mu = params[:, 0]                    # (batch,)
        raw = params[:, 1]                   # (batch,)

        # Convert unconstrained rho -> positive sigma (softplus is standard)
        sigma = F.softplus(raw) + self.min_sigma
        dist = Normal(loc=mu, scale=sigma)   # batch shape (batch,)
        return dist, kl_sum

Since the output of the model is a distribution, rather than a point estimate,
we use the [negative loglikelihood](https://en.wikipedia.org/wiki/Likelihood_function)
as our loss function to compute how likely to see the true data (targets) from the
estimated distribution produced by the model.

In [57]:
def train_probabilistic_bnn(
    model,
    train_dataloader,
    test_dataloader,
    num_epochs=100,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Practical default KL scaling: beta = kl_weight / N_train
    n_train = len(train_dataloader.dataset)
    beta = kl_weight / n_train

    def eval_rmse_using_mean(dataloader):
        model.eval()
        se_sum = 0.0
        n = 0
        with torch.no_grad():
            for X, y in dataloader:
                X = X.to(device)
                y = y.to(device)
                dist, _ = model(X)
                mu = dist.loc
                se_sum += torch.sum((mu - y) ** 2).item()
                n += y.numel()
        return np.sqrt(se_sum / n)

    print("Start training probabilistic BNN (NLL + beta*KL)...")
    for epoch in range(1, num_epochs + 1):
        model.train()
        nll_sum = 0.0
        kl_term_sum = 0.0
        total_sum = 0.0
        n_seen = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            dist, kl = model(X)

            nll = -dist.log_prob(y).mean()     # negative log likelihood
            loss = nll + beta * kl

            loss.backward()
            optimizer.step()

            bs = y.numel()
            nll_sum += nll.item() * bs
            kl_term_sum += (beta * kl).item() * bs
            total_sum += loss.item() * bs
            n_seen += bs

        if print_every is not None:
            if (epoch % print_every == 0) or (epoch == 1) or (epoch == num_epochs):
                train_rmse = eval_rmse_using_mean(train_dataloader)
                test_rmse = eval_rmse_using_mean(test_dataloader)
                print(
                    f"Epoch {epoch:3d}/{num_epochs} | "
                    f"loss={total_sum/n_seen:.4f} | nll={nll_sum/n_seen:.4f} | kl={kl_term_sum/n_seen:.5f} | "
                    f"train_rmse={train_rmse:.3f} | test_rmse={test_rmse:.3f}"
                )

    return model


In [58]:
num_epochs = 500

bnn_model_3 = ProbabilisticBayesianWineMLP(activation="sigmoid", posterior_rho_init=-2)
train_probabilistic_bnn(
    model=bnn_model_3,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=100,
)

Start training probabilistic BNN (NLL + beta*KL)...
Epoch   1/500 | loss=24.3486 | nll=24.3471 | kl=0.00151 | train_rmse=5.863 | test_rmse=5.823
Epoch 100/500 | loss=1.9182 | nll=1.9166 | kl=0.00159 | train_rmse=1.295 | test_rmse=1.176
Epoch 200/500 | loss=1.1648 | nll=1.1633 | kl=0.00158 | train_rmse=0.775 | test_rmse=0.696
Epoch 300/500 | loss=1.1532 | nll=1.1515 | kl=0.00168 | train_rmse=0.767 | test_rmse=0.699
Epoch 400/500 | loss=1.1313 | nll=1.1295 | kl=0.00180 | train_rmse=0.756 | test_rmse=0.696
Epoch 500/500 | loss=1.1152 | nll=1.1133 | kl=0.00192 | train_rmse=0.747 | test_rmse=0.689


ProbabilisticBayesianWineMLP(
  (bn): BatchNorm1d(11, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
  (fc1): LinearReparameterization()
  (fc2): LinearReparameterization()
  (param_head): Linear(in_features=8, out_features=2, bias=True)
  (act): Sigmoid()
)

Now let's produce an output from the model given the test examples.
The output is now a distribution, and we can use its mean and variance
to compute the confidence intervals (CI) of the prediction.

In [110]:
def predict_probabilistic_and_print(model, examples, targets, mc_samples=200):
    device = next(model.parameters()).device
    examples = examples.to(device)
    targets = targets.to(device)

    model.eval()

    mu_samples = []
    sigma_samples = []

    with torch.no_grad():
        for _ in range(mc_samples):
            dist, _ = model(examples)
            mu_samples.append(dist.loc)       # (batch,)
            sigma_samples.append(dist.scale)  # (batch,)

    mu_samples = torch.stack(mu_samples, dim=0)         # (S, B)
    sigma_samples = torch.stack(sigma_samples, dim=0)   # (S, B)

    # Decomposition: Var(y|x) = Var(mu) + E[sigma^2]
    pred_mean = mu_samples.mean(dim=0)                           # (B,)
    var_epistemic = mu_samples.var(dim=0, unbiased=False)        # (B,)
    var_aleatoric = (sigma_samples ** 2).mean(dim=0)             # (B,)
    total_std = torch.sqrt(var_epistemic + var_aleatoric)        # (B,)

    # Prefer posterior predictive intervals via sampling
    y_samples = mu_samples + sigma_samples * torch.randn_like(mu_samples)  # (S, B)
    lower = y_samples.quantile(0.025, dim=0)
    upper = y_samples.quantile(0.975, dim=0)

    # Move to CPU for printing
    pred_mean = pred_mean.cpu()
    total_std = total_std.cpu()
    lower = lower.cpu()
    upper = upper.cpu()
    targets = targets.cpu()

    for idx in range(len(pred_mean)):
        print(
            f"Prediction mean: {pred_mean[idx].item():.2f}, "
            f"stddev: {total_std[idx].item():.2f}, "
            f"95% PI: [{lower[idx].item():.2f}, {upper[idx].item():.2f}] "
            f"- Actual: {targets[idx].item():.1f}"
        )


sample = 10
examples, targets = next(iter(test_dataloader))
examples = examples[:sample]
targets = targets[:sample]

predict_probabilistic_and_print(
    bnn_model_3,
    examples,
    targets,
    mc_samples=200,
)

Prediction mean: 5.55, stddev: 0.74, 95% PI: [4.03, 6.83] - Actual: 5.0
Prediction mean: 5.55, stddev: 0.75, 95% PI: [4.34, 7.12] - Actual: 5.0
Prediction mean: 6.50, stddev: 0.84, 95% PI: [4.74, 8.14] - Actual: 7.0
Prediction mean: 6.69, stddev: 0.81, 95% PI: [5.10, 8.39] - Actual: 7.0
Prediction mean: 6.87, stddev: 0.81, 95% PI: [5.07, 8.53] - Actual: 8.0
Prediction mean: 6.77, stddev: 0.82, 95% PI: [5.43, 8.33] - Actual: 6.0
Prediction mean: 6.81, stddev: 0.82, 95% PI: [5.45, 8.36] - Actual: 7.0
Prediction mean: 5.94, stddev: 0.70, 95% PI: [4.64, 7.38] - Actual: 7.0
Prediction mean: 6.60, stddev: 0.83, 95% PI: [4.96, 8.21] - Actual: 5.0
Prediction mean: 6.18, stddev: 0.79, 95% PI: [4.72, 7.66] - Actual: 6.0
